# Inference runner: zero-shot (C1) and three-shot (C2)

Thin runner. All logic lives in `src/spider_sql`; this notebook only clones the repo, installs it,
unpacks the data bundle from Drive and calls the scripts. Runtime: GPU T4.

Rules (SPEC 3.2, review point 1): validation and dev only. The test split needs `--final` and runs
once per condition, after everything is frozen. Predictions are written to Drive and resume
automatically after a disconnect: rerun the same cell.

In [ ]:
from google.colab import drive

MOUNT = "/content/drive"  # the only path to edit if you mount elsewhere
drive.mount(MOUNT)

In [ ]:
!test -d /content/repo || git clone https://github.com/frangarciaproff/spider-sql-lora-ladder.git /content/repo
!git -C /content/repo pull --ff-only
%cd /content/repo
!pip install -q -e .
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

DRIVE = f"{MOUNT}/MyDrive/fine-tuning"
# writing to an unmounted path silently uses the runtime disk, which is lost
assert os.path.ismount(MOUNT), f"{MOUNT} is not a mounted Drive"
assert os.path.isfile(f"{DRIVE}/spider_bundle.zip"), f"no bundle in {DRIVE}: wrong account?"
# data version pinned in SPEC.md 3.1
BUNDLE_SHA256 = "aacfdd35b1a6afc369ef028cff59c39ac8639fd813c3b092fb7e5a1615e2b44f"
!python scripts/get_data.py --zip {DRIVE}/spider_bundle.zip --dest /content/data --sha256 {BUNDLE_SHA256}
os.environ["SPIDER_DIR"] = "/content/data/spider"
os.environ["TESTSUITE_DIR"] = "/content/data/testsuite"

## Configuration
Change `MODEL`, `TAG` and `BATCH` for the 1.5B run (`Qwen/Qwen2.5-Coder-1.5B-Instruct`, `qwen15b`, 16).

In [ ]:
import torch

assert torch.cuda.is_available(), (
    f"No usable GPU (torch {torch.__version__}, CUDA {torch.version.cuda}). "
    "Runtime > Change runtime type > T4; if T4 is refused, the free quota is used up."
)
print(torch.cuda.get_device_name(0))

MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
TAG = "qwen05b"
BATCH = 32
PRED = f"{DRIVE}/predictions"
RES = f"{DRIVE}/results"

## 1. Smoke test (32 dev examples, partial score, never reported)
Check that the outputs look like SQL. If they are empty or garbage, stop: fp16 may be unstable,
rerun with `--dtype float32` and tell Claude.

In [ ]:
!python scripts/generate.py --model {MODEL} --condition zero_shot --split dev --limit 32 --batch-size {BATCH} --out-dir {PRED}/smoke_{TAG}_c1
!python scripts/score_predictions.py --pred-dir {PRED}/smoke_{TAG}_c1 --split dev --out-dir {RES}/smoke_{TAG}_c1 --partial
!head -c 1500 {PRED}/smoke_{TAG}_c1/dev.jsonl

## 2. C1 zero-shot on validation and dev

In [ ]:
for split in ["validation", "dev"]:
    run = f"{TAG}_c1_zero_shot"
    !python scripts/generate.py --model {MODEL} --condition zero_shot --split {split} --batch-size {BATCH} --out-dir {PRED}/{run}
    !python scripts/score_predictions.py --pred-dir {PRED}/{run} --split {split} --out-dir {RES}/{run}

## 3. C2 three-shot on validation and dev

In [ ]:
for split in ["validation", "dev"]:
    run = f"{TAG}_c2_three_shot"
    !python scripts/generate.py --model {MODEL} --condition three_shot --split {split} --batch-size {BATCH} --out-dir {PRED}/{run}
    !python scripts/score_predictions.py --pred-dir {PRED}/{run} --split {split} --out-dir {RES}/{run}

## 4. Copy results into the repo
Download `results/<run>/` from Drive into the local repo's `results/` folder and commit it.
Predictions stay on Drive (they include raw model output).